[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/097.%20The%20Last-Mile%20Delivery%20%26%20Micro-Fulfillment%20Problem/P97-Tier-9_executed.ipynb)

---

# 97. The Last-Mile Delivery and Micro-Fulfillment Problem
## Tier 9: QUBO and Quantum-Inspired Annealing

### Purpose and scope
This notebook builds a Quadratic Unconstrained Binary Optimization (QUBO) model for assigning packages to vehicles, then searches for a low-energy binary assignment with a simulated annealing procedure. It also compares the resulting assignment with a greedy classical baseline and visualizes selected diagnostics.

The example uses a randomly generated, scaled scenario based on the stated Christmas Eve instance size. The code does **not** implement QAOA, execute a quantum circuit, or run on quantum hardware. Simulated annealing is a classical stochastic search procedure; describing it accurately is important when interpreting its results.

### Inputs and outputs
- **Inputs:** package and micro-fulfillment-center (MFC) coordinates, package demands, vehicle capacities, and the annealing parameters.
- **Outputs:** a QUBO energy model, a candidate package-to-vehicle assignment, assignment and load diagnostics, a greedy baseline comparison, and plots.
- **Decision variables:** one binary variable for each package–vehicle pair.

### Modeling assumptions and limitations
- Each package is assigned to one vehicle in the QUBO through a penalty for violating the one-vehicle-per-package condition.
- The distance term is the package's distance to its nearest MFC. It does not depend on the vehicle assignment, so this model does not choose an MFC or construct vehicle routes.
- Capacity is represented by penalties for an individually oversized package and for a pair of packages whose combined demand exceeds a vehicle's capacity. This pairwise QUBO surrogate does **not** guarantee that the total load of every vehicle is within capacity when three or more packages are considered together. Loads are therefore reported as diagnostics, not as guaranteed feasible constraints.
- The generated example may have total demand greater than total fleet capacity. In that case, assigning every package while respecting all capacities is infeasible, regardless of the optimization procedure.
- Time windows, travel times, vehicle start locations, route sequencing, and service times are not included in the mathematical model.

The scenario description mentions a much larger instance, but the scaled experiment does not establish quantum speedup, optimality, or full-scale performance. Such conclusions require appropriate hardware or algorithm implementations and independently measured results.

## 1. Background: binary assignments and QUBO

A binary decision variable can take only the values zero and one. Here, $x_{iv}=1$ means package $i$ is assigned to vehicle $v$, while $x_{iv}=0$ means it is not. A QUBO objective is a quadratic polynomial in binary variables; its minimizer is the assignment with the smallest modeled energy.

Let $i \in \{0,\ldots,n-1\}$ index packages and $v \in \{0,\ldots,V-1\}$ index vehicles. Let $d_i$ be package $i$'s demand, $C_v$ be vehicle $v$'s capacity, and $r_i$ be the distance from package $i$ to its nearest MFC. The model used in the code is

$$
E(x) = \sum_{i,v} r_i x_{iv}
      + A \sum_i \left(1-\sum_v x_{iv}\right)^2
      + B \sum_v \sum_i \max(0,d_i-C_v)^2 x_{iv}
      + B \sum_v \sum_{i<j} \mathbf{1}[d_i+d_j>C_v] x_{iv}x_{jv}.
$$

Here, $A$ is the assignment penalty weight, $B$ is the pairwise capacity-penalty weight, and $\mathbf{1}[\cdot]$ equals one when its condition is true and zero otherwise. The assignment penalty is zero exactly when a package has one selected vehicle; it is positive when the package is unassigned or assigned more than once. The constant term produced by expanding the square is omitted in the stored QUBO because adding or removing a constant does not change which solution minimizes the energy.

The last two terms penalize individual packages that exceed a vehicle's capacity and package pairs that cannot fit together. They are quadratic or linear in the binary variables, but they are only a pairwise capacity surrogate: several packages can collectively exceed capacity even when every individual package and every pair fits. The code later checks actual loads to make this limitation visible.

In matrix form, the energy is

$$
E(x) = x^{\mathsf T}Qx + h^{\mathsf T}x,
$$

where $Q$ contains quadratic coefficients, $h$ contains linear coefficients, and $x$ is the vector of all package–vehicle variables. The implementation stores both symmetric off-diagonal entries for each pairwise coefficient; their two contributions in $x^{\mathsf T}Qx$ give the intended coefficient for that pair.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
@dataclass
class QuantumDeliveryProblem:
    """Data for the package-to-vehicle assignment example."""

    num_packages: int
    num_vehicles: int
    num_mfcs: int
    package_locations: List[Tuple[float, float]]
    mfc_locations: List[Tuple[float, float]]
    vehicle_capacities: List[float]
    demand_weights: List[float]
    time_windows: List[Tuple[int, int]]


@dataclass
class QUBOFormulation:
    """Quadratic binary energy model and its variable-index mapping."""

    Q_matrix: np.ndarray
    h_vector: np.ndarray
    num_variables: int
    mapping: Dict[str, int]


class QuantumOptimizer:
    """Build a QUBO and search it with simulated annealing."""

    def __init__(self, problem: QuantumDeliveryProblem):
        self.problem = problem
        self.num_qubits = problem.num_packages * problem.num_vehicles
        self.qubo: Optional[QUBOFormulation] = None

    def create_qubo_formulation(self) -> QUBOFormulation:
        """Construct the binary assignment energy described above."""
        problem = self.problem
        if problem.num_packages <= 0 or problem.num_vehicles <= 0:
            raise ValueError("The problem must contain packages and vehicles.")
        if len(problem.package_locations) != problem.num_packages:
            raise ValueError("The number of package locations must match num_packages.")
        if len(problem.demand_weights) != problem.num_packages:
            raise ValueError("The number of package demands must match num_packages.")
        if len(problem.mfc_locations) != problem.num_mfcs or problem.num_mfcs <= 0:
            raise ValueError("The MFC locations must match a positive num_mfcs.")
        if len(problem.vehicle_capacities) != problem.num_vehicles:
            raise ValueError("The number of capacities must match num_vehicles.")
        if any(demand < 0 for demand in problem.demand_weights):
            raise ValueError("Package demands must be nonnegative.")
        if any(capacity < 0 for capacity in problem.vehicle_capacities):
            raise ValueError("Vehicle capacities must be nonnegative.")

        num_vars = problem.num_packages * problem.num_vehicles
        Q_matrix = np.zeros((num_vars, num_vars), dtype=float)
        h_vector = np.zeros(num_vars, dtype=float)

        # A stable mapping makes the mathematical variable x_iv easy to find in the vector.
        mapping = {}
        for i in range(problem.num_packages):
            for v in range(problem.num_vehicles):
                mapping[f"x_{i}_{v}"] = i * problem.num_vehicles + v

        distance_weight = 1.0
        assignment_weight = 10.0
        capacity_weight = 15.0

        # Each selected assignment pays the package's nearest-MFC distance.
        for i in range(problem.num_packages):
            package_x, package_y = problem.package_locations[i]
            nearest_mfc_distance = min(
                np.hypot(package_x - mfc_x, package_y - mfc_y)
                for mfc_x, mfc_y in problem.mfc_locations
            )
            for v in range(problem.num_vehicles):
                var = mapping[f"x_{i}_{v}"]
                h_vector[var] += distance_weight * nearest_mfc_distance

        # Expanding A(1 - sum_v x_iv)^2 gives -A on each linear term
        # and 2A on each unordered pair. Symmetric Q entries store that 2A.
        for i in range(problem.num_packages):
            package_vars = [
                mapping[f"x_{i}_{v}"] for v in range(problem.num_vehicles)
            ]
            for var in package_vars:
                h_vector[var] -= assignment_weight
            for position, var1 in enumerate(package_vars):
                for var2 in package_vars[position + 1:]:
                    Q_matrix[var1, var2] += assignment_weight
                    Q_matrix[var2, var1] += assignment_weight

        # Penalize individually oversized packages and pairwise capacity conflicts.
        for v in range(problem.num_vehicles):
            for i in range(problem.num_packages):
                var_i = mapping[f"x_{i}_{v}"]
                excess = max(
                    0.0,
                    problem.demand_weights[i] - problem.vehicle_capacities[v],
                )
                h_vector[var_i] += capacity_weight * excess**2

                for j in range(i + 1, problem.num_packages):
                    pair_demand = (
                        problem.demand_weights[i] + problem.demand_weights[j]
                    )
                    if pair_demand > problem.vehicle_capacities[v]:
                        var_j = mapping[f"x_{j}_{v}"]
                        Q_matrix[var_i, var_j] += capacity_weight
                        Q_matrix[var_j, var_i] += capacity_weight

        qubo = QUBOFormulation(Q_matrix, h_vector, num_vars, mapping)
        self.qubo = qubo
        return qubo

    def quantum_annealing_simulation(
        self,
        qubo: QUBOFormulation,
        temperature: float = 1.0,
        cooling_rate: float = 0.95,
        num_iterations: int = 1000,
    ) -> Tuple[np.ndarray, List[float]]:
        """Search the QUBO using a classical simulated annealing schedule."""
        if temperature <= 0:
            raise ValueError("Temperature must be positive.")
        if not 0 < cooling_rate <= 1:
            raise ValueError("Cooling rate must be in the interval (0, 1].")
        if num_iterations <= 0:
            raise ValueError("num_iterations must be positive.")

        current_solution = np.random.randint(0, 2, qubo.num_variables)
        current_energy = self.calculate_qubo_energy(current_solution, qubo)
        best_solution = current_solution.copy()
        best_energy = current_energy
        energy_history = []
        current_temperature = temperature

        for _ in range(num_iterations):
            # A proposal changes one bit, or occasionally several bits, in the same QUBO state.
            new_solution = self.quantum_inspired_update(
                current_solution, current_temperature
            )
            new_energy = self.calculate_qubo_energy(new_solution, qubo)
            energy_difference = new_energy - current_energy

            # The Metropolis rule sometimes accepts an uphill move, helping exploration early on.
            if energy_difference <= 0 or random.random() < np.exp(
                -energy_difference / current_temperature
            ):
                current_solution = new_solution
                current_energy = new_energy
                if current_energy < best_energy:
                    best_solution = current_solution.copy()
                    best_energy = current_energy

            energy_history.append(best_energy)
            current_temperature *= cooling_rate

        return best_solution, energy_history

    def quantum_inspired_update(
        self, solution: np.ndarray, temperature: float
    ) -> np.ndarray:
        """Propose a bit-flip move; occasional multi-bit moves broaden the search."""
        new_solution = solution.copy()
        if len(solution) == 0:
            return new_solution

        if len(solution) >= 2 and random.random() < 0.1:
            num_flips = random.randint(2, min(5, len(solution)))
            flip_indices = random.sample(range(len(solution)), num_flips)
            new_solution[flip_indices] = 1 - new_solution[flip_indices]
        else:
            flip_idx = random.randrange(len(solution))
            new_solution[flip_idx] = 1 - new_solution[flip_idx]
        return new_solution

    def calculate_qubo_energy(
        self, solution: np.ndarray, qubo: QUBOFormulation
    ) -> float:
        """Evaluate E(x) = x.T Q x + h.T x for one binary state."""
        if solution.shape != (qubo.num_variables,):
            raise ValueError("The solution length must match the QUBO variable count.")
        return float(solution @ qubo.Q_matrix @ solution + qubo.h_vector @ solution)

    def decode_solution(
        self, solution: np.ndarray, qubo: QUBOFormulation
    ) -> Dict[int, int]:
        """Convert selected binary variables into package-to-vehicle assignments."""
        assignments = {}
        for i in range(self.problem.num_packages):
            selected_vehicles = [
                v
                for v in range(self.problem.num_vehicles)
                if solution[qubo.mapping[f"x_{i}_{v}"]] == 1
            ]
            if selected_vehicles:
                assignments[i] = selected_vehicles[0]
        return assignments


print("QUBO model and simulated annealing implementation are ready.")

## 2. Create the scaled scenario

The instance generator uses a fixed seed so that the package coordinates, demands, time windows, and capacities can be reproduced. The scale factor controls the number of generated objects; it does not assert that a small instance has the same performance or solution quality as the full-size scenario.

In [ ]:
print("=" * 64)
print("SCALED CHRISTMAS EVE DELIVERY SCENARIO")
print("=" * 64)

# Retain the stated scale factor and apply minimum sizes for a useful example.
scale_factor = 0.01
num_packages = max(8, int(25000 * scale_factor))
num_vehicles = max(2, int(156 * scale_factor))
num_mfcs = max(3, int(347 * scale_factor))

print(f"Packages: {num_packages} (scale factor {scale_factor:.0%})")
print(f"Vehicles: {num_vehicles}")
print(f"Micro-fulfillment centers: {num_mfcs}")

# Seed both random-number generators because this cell uses both libraries.
np.random.seed(42)
random.seed(42)

package_locations = []
demand_weights = []
time_windows = []
cluster_centers = [(25, 25), (75, 25), (25, 75), (75, 75)]

for _ in range(num_packages):
    cluster_center = random.choice(cluster_centers)
    x = random.gauss(cluster_center[0], 10)
    y = random.gauss(cluster_center[1], 10)
    package_locations.append((max(5, min(95, x)), max(5, min(95, y))))
    demand_weights.append(random.uniform(0.5, 3.0))

    start_hour = random.randint(14, 20)
    end_hour = start_hour + random.randint(2, 4)
    time_windows.append((start_hour, min(23, end_hour)))

mfc_locations = cluster_centers[:num_mfcs]
vehicle_capacities = [random.uniform(50, 100) for _ in range(num_vehicles)]

problem = QuantumDeliveryProblem(
    num_packages=num_packages,
    num_vehicles=num_vehicles,
    num_mfcs=num_mfcs,
    package_locations=package_locations,
    mfc_locations=mfc_locations,
    vehicle_capacities=vehicle_capacities,
    demand_weights=demand_weights,
    time_windows=time_windows,
)

total_demand = sum(demand_weights)
total_capacity = sum(vehicle_capacities)
capacity_ratio = total_demand / total_capacity if total_capacity > 0 else float("inf")

print("\nScenario diagnostics:")
print(f"Total demand: {total_demand:.1f} units")
print(f"Average package demand: {np.mean(demand_weights):.2f} units")
print(f"Total fleet capacity: {total_capacity:.1f} units")
print(f"Demand / fleet capacity: {capacity_ratio:.1%}")
print(f"MFC coordinates: {mfc_locations}")
if total_demand > total_capacity:
    print("Capacity note: total demand exceeds total fleet capacity.")

## 3. Build the QUBO and inspect its size

There are $nV$ binary variables because each of the $n$ packages has one candidate variable for each of the $V$ vehicles. Exhaustively checking every binary vector would involve $2^{nV}$ states. This count describes the size of a brute-force search space; it is not a measured runtime and does not establish a quantum advantage. The QUBO matrix itself has $(nV)^2$ entries in this dense representation.

In [ ]:
optimizer = QuantumOptimizer(problem)
qubo = optimizer.create_qubo_formulation()

num_binary_states_log10 = qubo.num_variables * np.log10(2)

print("QUBO model:")
print(f"Binary variables: {qubo.num_variables}")
print(f"Q matrix dimensions: {qubo.Q_matrix.shape}")
print(f"Nonzero Q entries: {np.count_nonzero(qubo.Q_matrix)}")
print(f"Linear coefficients: {len(qubo.h_vector)}")
print(
    "Brute-force state count: "
    f"2^{qubo.num_variables} (approximately 10^{num_binary_states_log10:.1f})"
)
print("This count is a search-space size, not a runtime or speedup estimate.")

## 4. Search the QUBO with simulated annealing

Simulated annealing maintains a current binary state and proposes bit flips. A move that lowers the energy is accepted; a move that raises it may also be accepted with probability $\exp(-\Delta E/T)$, where $\Delta E$ is the energy increase and $T$ is the current temperature. Cooling reduces the probability of accepting uphill moves over time. The algorithm records the best state found, not a proof of global optimality.

The proposal includes occasional multi-bit flips, but this is a classical heuristic update. It does not simulate quantum superposition, entanglement, or a QAOA circuit. With $N$ binary variables and a dense matrix, the vectorized energy evaluation is $O(N^2)$ per proposal; the requested iteration count therefore affects the total computational work.

In [ ]:
initial_temperature = 10.0
cooling_rate = 0.995
num_iterations = 500

print("Simulated annealing parameters:")
print(f"Initial temperature: {initial_temperature}")
print(f"Cooling rate: {cooling_rate}")
print(f"Iterations: {num_iterations}")

start_time = time.perf_counter()
best_solution, energy_history = optimizer.quantum_annealing_simulation(
    qubo,
    temperature=initial_temperature,
    cooling_rate=cooling_rate,
    num_iterations=num_iterations,
)
annealing_time = time.perf_counter() - start_time
assignments = optimizer.decode_solution(best_solution, qubo)

print("\nSearch diagnostics:")
print(f"Elapsed time: {annealing_time:.3f} seconds")
print(f"Best QUBO energy: {energy_history[-1]:.3f}")
print(f"Packages with at least one selected vehicle: {len(assignments)} of {num_packages}")
print(f"Assignment coverage: {len(assignments) / num_packages:.1%}")

vehicle_loads = {v: 0.0 for v in range(num_vehicles)}
for package_id, vehicle_id in assignments.items():
    vehicle_loads[vehicle_id] += demand_weights[package_id]

print("\nVehicle load diagnostics:")
for vehicle_id, load in vehicle_loads.items():
    capacity = vehicle_capacities[vehicle_id]
    utilization = load / capacity if capacity > 0 else float("inf")
    print(
        f"Vehicle {vehicle_id}: {load:.1f} / {capacity:.1f} units "
        f"({utilization:.1%} utilization)"
    )

assignment_counts = np.zeros(num_packages, dtype=int)
for package_id in range(num_packages):
    assignment_counts[package_id] = sum(
        best_solution[qubo.mapping[f"x_{package_id}_{vehicle_id}"]]
        for vehicle_id in range(num_vehicles)
    )
print(f"Packages with exactly one selected vehicle: {np.count_nonzero(assignment_counts == 1)}")
print(f"Packages with no selected vehicle: {np.count_nonzero(assignment_counts == 0)}")
print(f"Packages with multiple selected vehicles: {np.count_nonzero(assignment_counts > 1)}")

## 5. Compare with a greedy baseline

The baseline processes larger-demand packages first and assigns a package to the least-loaded vehicle that can accommodate it. If no vehicle can accommodate a package, that package remains unassigned. This comparison is descriptive: the greedy baseline prioritizes its capacity check, while the QUBO includes a pairwise capacity surrogate and may produce overloaded vehicles. Their coverage and load diagnostics should be read alongside distance and elapsed time.

In [ ]:
def greedy_classical_assignment(problem: QuantumDeliveryProblem) -> Dict[int, int]:
    """Assign large-demand packages first when a vehicle has enough remaining capacity."""
    assignments = {}
    vehicle_loads = {v: 0.0 for v in range(problem.num_vehicles)}
    package_indices = sorted(
        range(problem.num_packages),
        key=lambda i: problem.demand_weights[i],
        reverse=True,
    )

    for package_id in package_indices:
        feasible_vehicles = [
            v
            for v in range(problem.num_vehicles)
            if vehicle_loads[v] + problem.demand_weights[package_id]
            <= problem.vehicle_capacities[v]
        ]
        if feasible_vehicles:
            best_vehicle = min(feasible_vehicles, key=lambda v: vehicle_loads[v])
            assignments[package_id] = best_vehicle
            vehicle_loads[best_vehicle] += problem.demand_weights[package_id]

    return assignments


def calculate_total_distance(
    assignments: Dict[int, int], problem: QuantumDeliveryProblem
) -> float:
    """Sum each assigned package's distance to its nearest MFC."""
    total_distance = 0.0
    for package_id in assignments:
        package_x, package_y = problem.package_locations[package_id]
        total_distance += min(
            np.hypot(package_x - mfc_x, package_y - mfc_y)
            for mfc_x, mfc_y in problem.mfc_locations
        )
    return total_distance


start_time = time.perf_counter()
classical_assignments = greedy_classical_assignment(problem)
classical_time = time.perf_counter() - start_time

qubo_distance = calculate_total_distance(assignments, problem)
greedy_distance = calculate_total_distance(classical_assignments, problem)

print("Greedy baseline comparison:")
print(f"Greedy elapsed time: {classical_time:.6f} seconds")
print(f"QUBO annealing elapsed time: {annealing_time:.3f} seconds")
print(f"Greedy assigned packages: {len(classical_assignments)} of {num_packages}")
print(f"QUBO assigned packages: {len(assignments)} of {num_packages}")
print(f"Greedy nearest-MFC distance for assigned packages: {greedy_distance:.2f}")
print(f"QUBO nearest-MFC distance for assigned packages: {qubo_distance:.2f}")
print(
    "Distance values are totals over each method's assigned packages; "
    "they are not directly comparable when coverage differs."
)

## 6. Visualize the search and assignment diagnostics

The plots show the best energy recorded during the annealing run, package assignments and unassigned packages, and a baseline comparison. The scaling panel shows only the number of states in a hypothetical exhaustive binary search as package count grows with the vehicle count held fixed. It is not a measured runtime curve and does not compare classical and quantum hardware.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
fig.suptitle("Package Assignment: QUBO Annealing Diagnostics", fontsize=15)

# The best-so-far curve cannot increase because it stores the best state encountered.
ax1 = axes[0, 0]
iterations = np.arange(1, len(energy_history) + 1)
ax1.plot(iterations, energy_history, color="navy", linewidth=1.8)
ax1.set_xlabel("Iteration")
ax1.set_ylabel("Best QUBO energy")
ax1.set_title("Best energy found during the search")
ax1.grid(True, alpha=0.3)

# Distinct markers identify MFCs, vehicle assignments, and unassigned packages.
ax2 = axes[0, 1]
vehicle_colors = plt.get_cmap("tab10")
for vehicle_id in range(num_vehicles):
    vehicle_packages = [
        package_id
        for package_id, assigned_vehicle in assignments.items()
        if assigned_vehicle == vehicle_id
    ]
    if vehicle_packages:
        points = np.array([package_locations[i] for i in vehicle_packages])
        ax2.scatter(
            points[:, 0],
            points[:, 1],
            s=28,
            alpha=0.7,
            color=vehicle_colors(vehicle_id),
            label=f"Vehicle {vehicle_id}",
        )

unassigned_packages = [
    i for i in range(num_packages) if i not in assignments
]
if unassigned_packages:
    points = np.array([package_locations[i] for i in unassigned_packages])
    ax2.scatter(
        points[:, 0],
        points[:, 1],
        marker="x",
        s=30,
        color="gray",
        label="Unassigned",
    )

mfc_points = np.array(mfc_locations)
ax2.scatter(
    mfc_points[:, 0],
    mfc_points[:, 1],
    marker="D",
    s=100,
    color="black",
    label="MFC",
)
ax2.set_xlabel("X coordinate (model units)")
ax2.set_ylabel("Y coordinate (model units)")
ax2.set_title("Package locations and decoded assignments")
ax2.set_xlim(0, 100)
ax2.set_ylim(0, 100)
ax2.grid(True, alpha=0.3)
ax2.legend()

# Coverage is shown with distance so that a lower distance is not read without context.
ax3 = axes[1, 0]
method_names = ["Greedy baseline", "QUBO annealing"]
coverage = [len(classical_assignments), len(assignments)]
distances = [greedy_distance, qubo_distance]
positions = np.arange(len(method_names))
width = 0.36
ax3.bar(positions - width / 2, coverage, width, label="Assigned packages")
ax3.set_ylabel("Assigned packages")
ax3.set_xticks(positions)
ax3.set_xticklabels(method_names)
ax3.set_title("Coverage and nearest-MFC distance")
ax3.grid(axis="y", alpha=0.3)
ax3_distance = ax3.twinx()
ax3_distance.bar(
    positions + width / 2,
    distances,
    width,
    color="darkorange",
    alpha=0.75,
    label="Distance for assigned packages",
)
ax3_distance.set_ylabel("Nearest-MFC distance (model units)")
coverage_handles, coverage_labels = ax3.get_legend_handles_labels()
distance_handles, distance_labels = ax3_distance.get_legend_handles_labels()
ax3.legend(
    coverage_handles + distance_handles,
    coverage_labels + distance_labels,
    loc="upper left",
    fontsize=8,
)

# For fixed vehicle count V, the binary state count is 2^(packages * V).
ax4 = axes[1, 1]
package_counts = np.arange(1, max(num_packages, 2) + 1)
log10_state_counts = package_counts * num_vehicles * np.log10(2)
ax4.plot(package_counts, log10_state_counts, color="purple", linewidth=2)
ax4.axvline(
    num_packages,
    color="black",
    linestyle="--",
    label=f"Example: {num_packages} packages",
)
ax4.set_xlabel("Number of packages (vehicles held fixed)")
ax4.set_ylabel("Log10 of exhaustive binary states")
ax4.set_title("Size of the binary search space")
ax4.grid(True, alpha=0.3)
ax4.legend()

fig.tight_layout()
plt.show()

## 7. Interpret the results

The energy history reports the best QUBO value encountered by the heuristic; it is not an optimality certificate. The decoded assignment should be interpreted together with the exact-one counts, coverage, vehicle loads, and total demand-versus-capacity diagnostic. Because the capacity terms are pairwise, a low-energy state can still overload a vehicle when the combined demand of several packages exceeds its capacity.

The nearest-MFC distance is the same package-level cost for every vehicle choice in this model. Consequently, it does not measure a vehicle route, account for a selected MFC, or establish that deliveries meet time windows. The greedy comparison is a baseline on this simplified assignment task, not evidence of quantum speedup or a general ranking of optimization methods.

### Further modeling extensions
Natural extensions include explicit MFC selection, route sequencing and travel costs, time-window feasibility, and a capacity formulation that represents total vehicle load. These additions require corresponding decision variables and penalties and are outside the model implemented here.